# Predictive Maintenance for Manufacturing

This notebook demonstrates the trained model and the error analysis produced by `scripts/train.py`. Run the training script first so the saved model and figures exist.

The task: flag the next operating day on which a machine is likely to fail, from nine anonymised sensor metrics and the calendar date.

In [ ]:
import json
import sys
from pathlib import Path

import joblib
import pandas as pd
from IPython.display import Image, display

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

from src import config
from src.data import build_features, load_dataset, prepare
from scripts.predict import score

pd.set_option('display.max_columns', 30)

## 1. Dataset

The supplied file contains one row per device-day. Failures are rare, so the split is stratified **by device**: every reading of a machine stays in the same split.

In [ ]:
frame = load_dataset(config.DATA_PATH)
print(f'rows={len(frame):,}  devices={frame[config.DEVICE].nunique():,}  '
      f'failures={frame[config.TARGET].sum()}  '
      f'failure_rate={frame[config.TARGET].mean():.4%}')
frame.head()

## 2. Features

Raw metrics, calendar fields, device age, and for each metric a z-score against the device's **past** readings only. `metric8` is dropped because it duplicates `metric7`. The expanding baseline is shifted by one day, so no future information enters a feature.

In [ ]:
features, target, devices, splits = prepare(frame)
print(f'{features.shape[1]} features:', ', '.join(features.columns))
for name, idx in splits.items():
    print(f'{name:<5} rows={len(idx):>6,}  failures={int(target.iloc[idx].sum()):>3}')

## 3. Results from the saved run

PR-AUC is the primary ranking metric because only about 0.08% of days are failures. The operating threshold is chosen on the validation split, then applied once to the test split.

In [ ]:
report = json.loads((config.OUTPUT_DIR / 'metrics.json').read_text(encoding='utf-8'))
print('selected model:', report['selected_model'])
print('threshold:', round(report['threshold'], 4))
print('test PR-AUC:', round(report['test_ranking']['pr_auc'], 4),
      '| ROC-AUC:', round(report['test_ranking']['roc_auc'], 4))
print('test at threshold:', report['test_at_threshold'])
pd.read_csv(config.COMPARISON_PATH)

In [ ]:
for name in ['pr_curves.png', 'confusion_matrix.png', 'feature_importance.png']:
    display(Image(filename=config.FIGURE_DIR / name))

## 4. Prototype: score readings

`score()` accepts a CSV with `device`, `date` (m/d/YYYY) and `metric1`…`metric9`. It rebuilds the same features and returns a probability plus a maintenance decision. Below we replay the last readings of a device that actually failed.

In [ ]:
bundle = joblib.load(config.MODEL_PATH)
failed_device = frame.loc[frame[config.TARGET] == 1, config.DEVICE].iloc[0]
history = frame[frame[config.DEVICE] == failed_device]
scores = score(history, bundle)
scores.tail(5)

In [ ]:
# To score your own file, save it with the columns listed above and run:
# my_scores = score(load_dataset('path/to/readings.csv'), bundle)
# my_scores.tail()
# Or from a terminal: python scripts/predict.py --input path/to/readings.csv